# 05 — Feature Performance by Category
## Genna Content Machine

Para cada uma das 62 features, uma tabela mostrando **métricas medianas** e **taxas de engajamento** por categoria.

- Colunas de **taxa** (gradiente vermelho→verde): Like Rate, Comment Rate, Share Rate, Save Rate, Interaction Rate
- Colunas de **volume** (gradiente azul): Reach, Likes, Comments, Shares, Saves, Interactions
- Taxas = métrica / reach (mediana por post em cada categoria)
- Linha **OVERALL** = baseline geral para comparação

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json
from IPython.display import display, HTML

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "general_dataset.parquet")

with open(PROJECT_ROOT / "general_features_reference.json", "r", encoding="utf-8") as f:
    features_ref = json.load(f)

print(f"Dataset: {df.shape[0]} posts x {df.shape[1]} colunas")
print(f"Features: {len([f for c in features_ref['categories'] for f in c['features']])} across {len(features_ref['categories'])} categories")

In [ ]:
NUMERIC_BINS = {
    "video_duration_sec": ([0, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 9999],
                           ["< 30s", "30-40s", "40-50s", "50-60s", "60-70s", "70-80s", "80-90s",
                            "90-100s", "100-110s", "110-120s", "120-130s", "130-140s", "140-150s", "> 150s"]),
    "carousel_slide_count": ([0, 5, 7, 10, 99], ["2-5", "6-7", "8-10", "> 10"]),
    "cuts_per_minute": ([0, 5, 10, 15, 20, 999], ["0-5", "5-10", "10-15", "15-20", "> 20"]),
    "speech_pacing_wpm": ([0, 120, 160, 200, 999], ["Slow (<120)", "Conversational (120-160)", "Fast (160-200)", "Rapid (>200)"]),
    "caption_length_chars": ([0, 50, 150, 500, 1000, 9999], ["Micro (<50)", "Short (50-150)", "Medium (150-500)", "Long (500-1K)", "Very Long (>1K)"]),
    "hashtag_count": ([-1, 0, 2, 5, 10, 99], ["0", "1-2", "3-5", "6-10", "> 10"]),
    "account_mentions_count": ([-1, 0, 1, 3, 99], ["0", "1", "2-3", "> 3"]),
}

SCORE_FEATURES = {
    "image_quality_score", "curiosity_gap_level", "first_frame_scroll_stop_score",
    "creator_camera_presence_score", "educational_density", "controversy_level",
    "overall_provocation_score",
}

MIN_N = 2

reels = df[df["post_type"] == "REEL"].copy()
carousels = df[df["post_type"] == "FEED_CAROUSEL"].copy()


def build_table(feature, data=None):
    if data is None:
        data = df
    sub = data[data[feature].notna() & data["reach"].notna() & (data["reach"] > 0)].copy()
    if len(sub) < 3:
        return None

    if feature in NUMERIC_BINS:
        bins, labels = NUMERIC_BINS[feature]
        sub["_grp"] = pd.cut(sub[feature], bins=bins, labels=labels, right=True)
    elif feature in SCORE_FEATURES:
        sub["_grp"] = sub[feature].astype(int).astype(str)
    else:
        sub["_grp"] = sub[feature].astype(str)

    sub = sub.dropna(subset=["_grp"])

    for m in ["likes", "comments", "shares", "saved", "interactions"]:
        sub[f"_r_{m}"] = sub[m] / sub["reach"] * 100

    def agg(g):
        return {
            "n": len(g),
            "Reach": g["reach"].median(),
            "Likes": g["likes"].median(),
            "Like Rate": g["_r_likes"].median(),
            "Comments": g["comments"].median(),
            "Comment Rate": g["_r_comments"].median(),
            "Shares": g["shares"].median(),
            "Share Rate": g["_r_shares"].median(),
            "Saves": g["saved"].median(),
            "Save Rate": g["_r_saved"].median(),
            "Interactions": g["interactions"].median(),
            "Interaction Rate": g["_r_interactions"].median(),
        }

    rows = {}
    for cat, g in sub.groupby("_grp", observed=True):
        if len(g) < MIN_N:
            continue
        rows[str(cat)] = agg(g)

    if not rows:
        return None

    rows["OVERALL"] = agg(sub)

    result = pd.DataFrame.from_dict(rows, orient="index")
    result.index.name = feature

    overall = result.loc[["OVERALL"]]
    rest = result.drop("OVERALL")

    if feature in SCORE_FEATURES:
        rest = rest.loc[sorted(rest.index, key=lambda x: int(float(x)))]
    else:
        rest = rest.sort_values("Reach", ascending=False)

    return pd.concat([rest, overall])


def style_table(t):
    rate_cols = [c for c in t.columns if "Rate" in c]
    abs_cols = ["Reach", "Likes", "Comments", "Shares", "Saves", "Interactions"]
    abs_cols = [c for c in abs_cols if c in t.columns]

    fmt = {
        "n": "{:.0f}",
        "Reach": "{:,.0f}",
        "Likes": "{:,.0f}",
        "Comments": "{:,.0f}",
        "Shares": "{:,.0f}",
        "Saves": "{:,.0f}",
        "Interactions": "{:,.0f}",
    }
    for c in rate_cols:
        fmt[c] = "{:.2f}%"

    styled = (
        t.style
        .background_gradient(cmap="RdYlGn", subset=rate_cols)
        .background_gradient(cmap="Blues", subset=abs_cols)
        .format(fmt)
    )

    if "OVERALL" in t.index:
        styled = styled.set_properties(
            subset=pd.IndexSlice["OVERALL", :],
            **{"font-weight": "bold", "border-top": "2px solid #333"}
        )

    return styled

print("Helpers loaded.")

In [ ]:
def build_description_html(feat_info):
    applies = feat_info.get("applies_to", "")
    pv = feat_info.get("possible_values", "")
    parts = [s.strip() for s in pv.split(";")]
    cat_rows = []
    general_desc = None
    for p in parts:
        if " — " in p:
            val, desc = p.split(" — ", 1)
            cat_rows.append(f"<tr><td style='padding:2px 10px;font-weight:600'>{val.strip()}</td>"
                            f"<td style='padding:2px 10px;color:#555'>{desc.strip()}</td></tr>")
        elif p:
            if general_desc is None:
                general_desc = p
            else:
                general_desc += "; " + p

    html = f"<div style='margin:8px 0 12px 0;padding:10px 14px;background:#f8f9fa;border-left:3px solid #2c3e50;border-radius:4px;font-size:13px'>"
    html += f"<b>Applies to:</b> {applies}"
    if general_desc:
        html += f" &nbsp;|&nbsp; <b>Type:</b> {general_desc}"
    if cat_rows:
        html += f"<table style='margin-top:6px;border-collapse:collapse'>{''.join(cat_rows)}</table>"
    html += "</div>"
    return html


def display_segment_table(label, data, fname):
    table = build_table(fname, data)
    if table is None:
        return False
    n_cats = table.shape[0] - 1
    n_posts = int(table.loc[table.index != "OVERALL", "n"].sum())
    display(HTML(f"<h4 style='margin:12px 0 4px 0;color:#34495e'>{label} "
                 f"<small style='color:#999'>({n_cats} categorias, {n_posts} posts)</small></h4>"))
    display(style_table(table))
    return True


total = 0
skipped = 0

for cat in features_ref["categories"]:
    display(HTML(f"<hr><h2 style='color:#2c3e50; margin-top:40px'>{cat['name']}</h2>"))

    for feat_info in cat["features"]:
        fname = feat_info["name"]
        applies = feat_info.get("applies_to", "")
        total += 1

        if fname not in df.columns:
            skipped += 1
            continue

        show_reels = "R" in applies
        show_carousels = "C" in applies or "I" in applies

        displayed = False

        display(HTML(f"<h3 style='margin-top:28px'>{fname}</h3>"))
        display(HTML(build_description_html(feat_info)))

        if show_reels:
            if display_segment_table("Reels", reels, fname):
                displayed = True
        if show_carousels:
            if display_segment_table("Carrosseis", carousels, fname):
                displayed = True

        if not displayed:
            display(HTML(f"<p><em>dados insuficientes</em></p>"))
            skipped += 1

print(f"\n{'='*60}")
print(f"Total: {total} features | Exibidas: {total - skipped} | Sem dados: {skipped}")